# SSL masked reconstruction pretraining, ps32

This notebook pretrains a modified ResNet-18 encoder with masked reconstruction on unlabeled 14-channel raster patches. It does not use landslide/non-landslide labels, does not fine-tune a classifier, and does not generate susceptibility maps.

## 1. Purpose and experiment configuration

This is in-domain transductive SSL pretraining: unlabeled patches are sampled from the whole cleaned study area. SCV holdout clusters are not excluded because no landslide/non-landslide labels are used during SSL pretraining.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

patch_size = 32
n_unlabeled_patches = 20000
normalization_sample_size = 5000
random_seed = 42
batch_size = 64
learning_rate = 1e-4
weight_decay = 1e-4
max_epochs = 50
early_stopping_patience = 10
mask_ratio = 0.5
block_size = 4
gradient_clip_norm = 5.0

raster_dir = PROJECT_ROOT / "data/processed/rasters_cleaned"
unlabeled_index_csv = PROJECT_ROOT / "data/processed/ssl_unlabeled_indices/unlabeled_patch_index_ps32_n20000.csv"
output_root = PROJECT_ROOT / "outputs/SSL_masked_recon_ps32"
figure_root = PROJECT_ROOT / "figures/SSL_masked_recon_ps32"
checkpoint_dir = PROJECT_ROOT / "checkpoints/ssl_pretrained/masked_recon"

## 2. Import packages and project modules

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, random_split

from src.patch_dataset import DEFAULT_NODATA_VALUE, audit_raster_alignment, list_raster_files
from src.plotting import plot_masked_reconstruction_examples, plot_ssl_loss_curve
from src.ssl_masked_recon import (
    MaskedReconstructionModel,
    MaskedReconstructionRasterDataset,
    compute_ssl_channel_stats,
    create_unlabeled_patch_index,
    generate_block_mask,
)
from src.train_ssl import train_masked_reconstruction_model
from src.utils import count_trainable_parameters, ensure_dir, get_device, set_global_seed

## 3. Set random seeds and device

In [ ]:
set_global_seed(random_seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = torch.cuda.is_available()
num_workers = 0
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"CUDA GPU: {torch.cuda.get_device_name(0)}")
    print(f"torch.version.cuda: {torch.version.cuda}")
else:
    print("WARNING: CUDA is not available; training will run on CPU.")

## 4. Load and audit cleaned rasters

In [ ]:
raster_files = list_raster_files(raster_dir)
raster_audit = audit_raster_alignment(raster_files, expected_nodata=DEFAULT_NODATA_VALUE)
print(f"number of raster files: {len(raster_files)}")
print(f"raster shape: {raster_audit.height} x {raster_audit.width}")
print(f"nodata: {raster_audit.nodata}")

## 5. Generate or load unlabeled patch index

In [ ]:
unlabeled_index = create_unlabeled_patch_index(
    raster_dir=raster_dir,
    output_csv=unlabeled_index_csv,
    patch_size=patch_size,
    n_patches=n_unlabeled_patches,
    nodata_value=DEFAULT_NODATA_VALUE,
    max_nodata_ratio=0.0,
    random_seed=random_seed,
    max_attempts=1_000_000,
)
print(f"number of valid unlabeled patches obtained: {len(unlabeled_index)}")
print(f"unlabeled index: {unlabeled_index_csv}")

## 6. Compute SSL channel normalization statistics

In [ ]:
raw_ssl_dataset = MaskedReconstructionRasterDataset(
    patch_index_csv=unlabeled_index_csv,
    raster_dir=raster_dir,
    patch_size=patch_size,
    normalize=False,
    return_metadata=False,
)
channel_means, channel_stds = compute_ssl_channel_stats(
    raw_ssl_dataset,
    sample_size=normalization_sample_size,
    batch_size=batch_size,
    random_seed=random_seed,
)
print(f"channel_means shape: {channel_means.shape}")
print(f"channel_stds shape: {channel_stds.shape}")
print(channel_means)
print(channel_stds)

## 7. Define masked reconstruction Dataset and DataLoaders

In [ ]:
ssl_dataset = MaskedReconstructionRasterDataset(
    patch_index_csv=unlabeled_index_csv,
    raster_dir=raster_dir,
    patch_size=patch_size,
    normalize=True,
    channel_means=channel_means,
    channel_stds=channel_stds,
    return_metadata=False,
)
X0 = raw_ssl_dataset[0]
X0_norm = ssl_dataset[0]
print(f"X.shape: {tuple(X0.shape)}")
print(f"raw finite: {torch.isfinite(X0).all().item()}")
print(f"raw has nodata: {(X0 == DEFAULT_NODATA_VALUE).any().item()}")
print(f"normalized finite: {torch.isfinite(X0_norm).all().item()}")

train_size = int(0.9 * len(ssl_dataset))
val_size = len(ssl_dataset) - train_size
train_dataset, val_dataset = random_split(
    ssl_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(random_seed),
)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=pin_memory)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin_memory)
print(f"train/val split sizes: {train_size}, {val_size}")

## 8. Define modified ResNet-18 encoder and reconstruction decoder

In [ ]:
model = MaskedReconstructionModel(in_channels=14, out_channels=14).to(device)
print(f"model parameter count: {count_trainable_parameters(model)}")

## 9. Define masking strategy and reconstruction loss

In [ ]:
batch = next(iter(train_loader))
X = batch.to(device, non_blocking=True)
mask = generate_block_mask(X.shape[0], X.shape[2], X.shape[3], mask_ratio, block_size, device=X.device)
print(f"model device: {next(model.parameters()).device}")
print(f"X.device: {X.device}")
print(f"mask.device: {mask.device}")

## 10. Train masked reconstruction SSL model

In [ ]:
training_log_dir = ensure_dir(output_root / "training_logs")
recon_fig_dir = ensure_dir(figure_root / "reconstruction_examples")
curve_fig_dir = ensure_dir(figure_root / "training_curves")
checkpoint_dir = ensure_dir(checkpoint_dir)

config = {
    "patch_size": patch_size,
    "n_unlabeled_patches": n_unlabeled_patches,
    "mask_ratio": mask_ratio,
    "block_size": block_size,
    "batch_size": batch_size,
    "learning_rate": learning_rate,
    "weight_decay": weight_decay,
    "max_epochs": max_epochs,
    "early_stopping_patience": early_stopping_patience,
    "random_seed": random_seed,
}
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
full_model_best_path = checkpoint_dir / "resnet18_masked_recon_ps32_full_model_best.pt"
encoder_best_path = checkpoint_dir / "resnet18_masked_recon_ps32_encoder_best.pt"
last_checkpoint_path = checkpoint_dir / "resnet18_masked_recon_ps32_last.pt"

print(f"batch size: {batch_size}")
print(f"device: {device}")
print(f"mask_ratio: {mask_ratio}")
print(f"block_size: {block_size}")
print(f"max_epochs: {max_epochs}")
print(f"learning rate: {learning_rate}")
print(f"encoder checkpoint: {encoder_best_path}")
print(f"full model checkpoint: {full_model_best_path}")

training_log_df, best_info = train_masked_reconstruction_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    full_model_best_path=full_model_best_path,
    encoder_best_path=encoder_best_path,
    last_checkpoint_path=last_checkpoint_path,
    config=config,
    channel_means=channel_means,
    channel_stds=channel_stds,
    max_epochs=max_epochs,
    early_stopping_patience=early_stopping_patience,
    mask_ratio=mask_ratio,
    block_size=block_size,
    grad_clip_norm=gradient_clip_norm,
)

## 11. Save best pretrained encoder and logs

In [ ]:
training_log_path = training_log_dir / "masked_recon_ps32_training_log.csv"
training_log_df.to_csv(training_log_path, index=False)
print(f"training log: {training_log_path}")

## 12. Visualize reconstruction examples

In [ ]:
best_checkpoint = torch.load(full_model_best_path, map_location=device)
model.load_state_dict(best_checkpoint["model_state_dict"])
model.eval()
with torch.no_grad():
    X_vis = next(iter(val_loader)).to(device, non_blocking=True)
    mask_vis = generate_block_mask(X_vis.shape[0], X_vis.shape[2], X_vis.shape[3], mask_ratio, block_size, device=X_vis.device)
    X_recon_vis = model(X_vis * (1.0 - mask_vis))
recon_fig_path = recon_fig_dir / "masked_recon_examples_epoch_best.png"
plot_masked_reconstruction_examples(X_vis, mask_vis, X_recon_vis, recon_fig_path)
loss_curve_path = curve_fig_dir / "masked_recon_ps32_loss_curve.png"
plot_ssl_loss_curve(training_log_df, loss_curve_path)
print(f"reconstruction figure: {recon_fig_path}")
print(f"loss curve: {loss_curve_path}")

## 13. Print final summary and next-step notes

In [ ]:
print(f"number of valid unlabeled patches: {len(ssl_dataset)}")
print(f"best epoch: {best_info['best_epoch']}")
print(f"best validation loss: {best_info['best_val_loss']:.6f}")
print(f"final train loss: {training_log_df['train_loss'].iloc[-1]:.6f}")
print(f"final val loss: {training_log_df['val_loss'].iloc[-1]:.6f}")
print(f"encoder checkpoint path: {encoder_best_path}")
print(f"full model checkpoint path: {full_model_best_path}")
print(f"training log path: {training_log_path}")
print(f"reconstruction figure path: {recon_fig_path}")
print("Next step: use the encoder checkpoint in supervised SCV fine-tuning notebook.")